### Data Ingestion (Silver)

In [0]:
from pyspark.sql import SparkSession
from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = (date.today() - timedelta(days=1)).strftime('%Y-%m-%d')
path = f's3://ruturaj-serverless/bank_project/silver/transactions/{yesterday}/'

df = spark.read.format('delta').load(path)
transactions = df

### Data Transformation

In [0]:
# transactions.display()

daily_status_report = transactions.groupBy('status').agg(count('*').alias('count'))
daily_location_report = transactions.groupBy('location').agg(count('*').alias('count'))
daily_payment_type_report = transactions.groupBy('payment_type').agg(count('*').alias('count'))
daily_cutomer_report = transactions.groupBy('customer_id').agg(count('*').alias('total_orders'), sum('amount').alias('turnover')).orderBy('customer_id')


### Data Loading (Gold)

In [0]:
path = f's3://ruturaj-serverless/bank_project/gold/{yesterday}'

daily_status_report.write.format('delta').mode('overwrite').save(f'{path}/daily_status_report')
daily_location_report.write.format('delta').mode('overwrite').save(f'{path}/daily_location_report')
daily_payment_type_report.write.format('delta').mode('overwrite').save(f'{path}/daily_payment_type_report')
daily_cutomer_report.write.format('delta').mode('overwrite').save(f'{path}/daily_cutomer_report')